# NTS0611 Behaviour-to-POI Mapping

This notebook transforms harmonised trip purpose probabilities generated from the NTS0611 dataset into POI-based behavioural probabilities using the proposed semantic mapping framework.

**Input**

- `nts0611_trip_purpose_probability.json`

**Output**

- `nts0611_poi_probability.json`

### 1. Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import json

### 2. Define File Paths

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

INPUT_DIR = PROJECT_ROOT / "data" / "nts" / "harmonised"

OUTPUT_DIR = PROJECT_ROOT / "data" / "nts" / "harmonised"

INPUT_FILE = INPUT_DIR / "nts0611_trip_purposes.json"

OUTPUT_FILE = OUTPUT_DIR / "nts0611_poi_representation.json"

### 3. Load Trip Purpose Probability

In [3]:
trip_purpose_probability = pd.read_json(INPUT_FILE)

display(trip_purpose_probability.head())

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1]
0,2024,Females,0-4,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
1,2024,Females,5-11,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
2,2024,Females,12-17,0.026672,0.001690,0.304515,0.038551,0.056038,0.125005,0.074419,0.108590,0.051023,0.074196,0.031747,0.015177,0.040354,0.052022
3,2024,Females,18-22,0.180319,0.015662,0.117592,0.019347,0.189261,0.036292,0.071061,0.109007,0.080139,0.057312,0.007461,0.017746,0.035592,0.063210
4,2024,Females,23-34,0.187671,0.030108,0.019513,0.088612,0.184447,0.069928,0.074421,0.086093,0.055916,0.059392,0.006704,0.017174,0.035667,0.084355


### 4. Initialise POI Probability Matrix

In [4]:
# ==========================================
# Define POI Categories
# ==========================================

poi_categories = [
    "Employment",
    "Education",
    "Retail",
    "Healthcare",
    "Personal Services",
    "Public Services",
    "Food and Beverage",
    "Recreation",
    "Accommodation"
]

In [5]:
# ==========================================
# Initialise POI Probability Matrix
# ==========================================

poi_probability = trip_purpose_probability[
    ["Year", "Sex [note 2]", "Age"]
].copy()

for poi in poi_categories:
    poi_probability[poi] = 0.0

display(poi_probability.head())

,Year,Sex [note 2],Age,Employment,Education,Retail,Healthcare,Personal Services,Public Services,Food and Beverage,Recreation,Accommodation
0,2024,Females,0-4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024,Females,5-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024,Females,12-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024,Females,18-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024,Females,23-34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 5. Define Mapping Strategy

#### 5.1 Define Mapping Strategies

In [6]:
mapping_strategy_df = pd.DataFrame({

    "Mapping Strategy": [
        "Direct Mapping",
        "Representative Mapping",
        "Context-dependent Mapping",
        "Excluded Mapping"
    ],

    "Definition": [

        "A trip purpose has a clear one-to-one semantic correspondence with a single POI category.",

        "A trip purpose may involve multiple destinations in reality, but one representative POI category is adopted to maintain semantic consistency.",

        "A trip purpose cannot be resolved using semantic information alone. Multiple candidate POI categories are retained, and the final destination is determined later using contextual information.",

        "A trip purpose is excluded because it falls outside the scope of the POI dataset or lacks sufficient semantic information for destination inference."
    ]
})

display(mapping_strategy_df)

,Mapping Strategy,Definition
0,Direct Mapping,A trip purpose has a clear one-to-one semantic...
1,Representative Mapping,A trip purpose may involve multiple destinatio...
2,Context-dependent Mapping,A trip purpose cannot be resolved using semant...
3,Excluded Mapping,A trip purpose is excluded because it falls ou...


#### 5.2 Define Trip Purpose to POI Mapping

In [7]:
# ==========================================
# Define Trip Purpose to POI Mapping
# ==========================================

trip_purpose_mapping_df = pd.DataFrame(

    [

        # Representative Mapping
        ["Commuting", "Representative Mapping", "Employment"],
        ["Business", "Representative Mapping", "Employment"],
        ["Visiting friends elsewhere", "Representative Mapping", "Food and Beverage"],
        ["Entertainment or public activity", "Representative Mapping", "Recreation"],
        ["Day trip", "Representative Mapping", "Recreation"],
        ["Just walk [note 1]", "Representative Mapping", "Recreation"],

        # Direct Mapping
        ["Education", "Direct Mapping", "Education"],
        ["Escort education", "Direct Mapping", "Education"],
        ["Shopping", "Direct Mapping", "Retail"],
        ["Sport: participate", "Direct Mapping", "Recreation"],
        ["Holiday: base", "Direct Mapping", "Accommodation"],

        # Context-dependent Mapping
        [
            "Personal business",
            "Context-dependent Mapping",
            "Healthcare / Public Services / Personal Services"
        ],

        [
            "Other escort",
            "Context-dependent Mapping",
            "Employment / Retail / Healthcare / Personal Services / Public Services / Food and Beverage / Recreation / Accommodation"
        ],

        # Excluded Mapping
        [
            "Visiting friends at private home",
            "Excluded Mapping",
            "Private Residence (Outside POI Scope)"
        ]

    ],

    columns=[
        "Unified Trip Purpose",
        "Mapping Strategy",
        "POI"
    ]

)

display(trip_purpose_mapping_df)

,Unified Trip Purpose,Mapping Strategy,POI
0,Commuting,Representative Mapping,Employment
1,Business,Representative Mapping,Employment
2,Visiting friends elsewhere,Representative Mapping,Food and Beverage
3,Entertainment or public activity,Representative Mapping,Recreation
4,Day trip,Representative Mapping,Recreation
5,Just walk [note 1],Representative Mapping,Recreation
6,Education,Direct Mapping,Education
7,Escort education,Direct Mapping,Education
8,Shopping,Direct Mapping,Retail
9,Sport: participate,Direct Mapping,Recreation


### 6. Apply Mapping

#### 6.1 Direct Mapping

In [8]:
# ==========================================
# Apply Direct Mapping
# ==========================================

direct_mapping = trip_purpose_mapping_df[
    trip_purpose_mapping_df["Mapping Strategy"] == "Direct Mapping"
]

for _, row in direct_mapping.iterrows():

    trip_purpose = row["Unified Trip Purpose"]
    poi = row["POI"]

    poi_probability[poi] += trip_purpose_probability[trip_purpose]

display(poi_probability.head())

,Year,Sex [note 2],Age,Employment,Education,Retail,Healthcare,Personal Services,Public Services,Food and Beverage,Recreation,Accommodation
0,2024,Females,0-4,0.0,0.371659,0.031882,0.0,0.0,0.0,0.0,0.035980,0.015151
1,2024,Females,5-11,0.0,0.371659,0.031882,0.0,0.0,0.0,0.0,0.035980,0.015151
2,2024,Females,12-17,0.0,0.343067,0.056038,0.0,0.0,0.0,0.0,0.031747,0.015177
3,2024,Females,18-22,0.0,0.136939,0.189261,0.0,0.0,0.0,0.0,0.007461,0.017746
4,2024,Females,23-34,0.0,0.108125,0.184447,0.0,0.0,0.0,0.0,0.006704,0.017174


### 6.2 Representative Mapping

In [9]:
# ==========================================
# Apply Representative Mapping
# ==========================================

representative_mapping = trip_purpose_mapping_df[
    trip_purpose_mapping_df["Mapping Strategy"] == "Representative Mapping"
]

for _, row in representative_mapping.iterrows():

    trip_purpose = row["Unified Trip Purpose"]
    poi = row["POI"]

    poi_probability[poi] += trip_purpose_probability[trip_purpose]

display(poi_probability.head())

,Year,Sex [note 2],Age,Employment,Education,Retail,Healthcare,Personal Services,Public Services,Food and Beverage,Recreation,Accommodation
0,2024,Females,0-4,0.004317,0.371659,0.031882,0.0,0.0,0.0,0.043625,0.208958,0.015151
1,2024,Females,5-11,0.004317,0.371659,0.031882,0.0,0.0,0.0,0.043625,0.208958,0.015151
2,2024,Females,12-17,0.028362,0.343067,0.056038,0.0,0.0,0.0,0.051023,0.198319,0.015177
3,2024,Females,18-22,0.195981,0.136939,0.189261,0.0,0.0,0.0,0.080139,0.163575,0.017746
4,2024,Females,23-34,0.217779,0.108125,0.184447,0.0,0.0,0.0,0.055916,0.186119,0.017174


#### 6.3 Context-dependent Mapping

In [10]:
# ==========================================
# Apply Context-dependent Mapping
# ==========================================

context_mapping = trip_purpose_mapping_df[
    trip_purpose_mapping_df["Mapping Strategy"] == "Context-dependent Mapping"
]

for _, row in context_mapping.iterrows():

    trip_purpose = row["Unified Trip Purpose"]

    candidate_pois = [
        poi.strip()
        for poi in row["POI"].split("/")
    ]

    for poi in candidate_pois:
        poi_probability[poi] += trip_purpose_probability[trip_purpose]

display(poi_probability.head())

,Year,Sex [note 2],Age,Employment,Education,Retail,Healthcare,Personal Services,Public Services,Food and Beverage,Recreation,Accommodation
0,2024,Females,0-4,0.146632,0.371659,0.174197,0.217889,0.217889,0.217889,0.185940,0.351274,0.157466
1,2024,Females,5-11,0.146632,0.371659,0.174197,0.217889,0.217889,0.217889,0.185940,0.351274,0.157466
2,2024,Females,12-17,0.153367,0.343067,0.181043,0.199424,0.199424,0.199424,0.176027,0.323324,0.140182
3,2024,Females,18-22,0.232273,0.136939,0.225553,0.107353,0.107353,0.107353,0.116431,0.199867,0.054038
4,2024,Females,23-34,0.287707,0.108125,0.254374,0.144348,0.144348,0.144348,0.125843,0.256047,0.087101


#### 6.4 Excluded Mapping

In [11]:
# ==========================================
# Apply Excluded Mapping
# ==========================================

excluded_mapping = trip_purpose_mapping_df[
    trip_purpose_mapping_df["Mapping Strategy"] == "Excluded Mapping"
]

display(excluded_mapping)

,Unified Trip Purpose,Mapping Strategy,POI
13,Visiting friends at private home,Excluded Mapping,Private Residence (Outside POI Scope)


### 7. Validation

In [12]:
# ==========================================
# Check Missing Values
# ==========================================

poi_probability.isna().sum()

Year                 0
Sex [note 2]         0
Age                  0
Employment           0
Education            0
Retail               0
Healthcare           0
Personal Services    0
Public Services      0
Food and Beverage    0
Recreation           0
Accommodation        0
dtype: int64

In [13]:
# ==========================================
# Check Duplicate Records
# ==========================================

poi_probability.duplicated().sum()

np.int64(0)

In [14]:
# ==========================================
# Check Row Sums
# ==========================================

row_sum = poi_probability[poi_categories].sum(axis=1)

display(row_sum)

0     2.040833
1     2.040833
2     1.915283
3     1.287159
4     1.552242
5     1.787531
6     1.590391
7     1.504941
8     1.435749
9     2.021037
10    2.021037
11    1.901972
12    1.261773
13    1.461307
14    1.790347
15    1.623439
16    1.586692
17    1.572457
dtype: float64

### 8. Export

In [15]:
# ==========================================
# Export POI Probability
# ==========================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

poi_probability.to_json(
    OUTPUT_FILE,
    orient="records",
    indent=4
)

print(f"POI probability matrix saved to:\n{OUTPUT_FILE}")

POI probability matrix saved to:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-representative-profile\data\nts\harmonised\nts0611_poi_representation.json


## Summary

**Input**

- `nts0611_trip_purpose_probability.json`

**Output**

- `nts0611_poi_probability.json`

**Processing Completed**

- Trip purpose probabilities loaded
- POI probability matrix initialised
- Semantic mapping applied
- Context-dependent candidates retained
- POI probability matrix exported